# Fase 1 — Entendimiento del negocio

## 1.1 Contexto y objetivo del negocio

High Garden Coffee es una exportadora internacional de café. Su área de innovación quiere usar datos para ganar ventaja
competitiva: identificar tendencias, anticipar el mercado y encontrar información accionable.

**Problema de negocio.** Una exportadora compite por *dónde* vender. Los compradores tradicionales (UE, EE. UU., Japón)
son mercados maduros y muy disputados. En paralelo, varios **países productores** consumen una parte cada vez mayor de su
propia cosecha. Cuando su consumo interno alcanza o supera su producción, pasan a necesitar café importado: se convierten en
**nuevos mercados de destino** que la competencia suele pasar por alto.

**Objetivo de negocio.** Identificar y priorizar, con un horizonte de **3 años cafeteros**, los países productores cuyo balance
producción − consumo se está cerrando o ya es negativo, para orientar la estrategia comercial de exportación.

**Preguntas de negocio**
1. ¿Qué países productores ya consumen más café del que producen (importadores netos)?
2. ¿En cuáles el consumo interno crece más rápido que la producción, y cuándo cruzarían la línea de déficit?
3. ¿Cuál es el tamaño esperado del déficit (en sacos) en los próximos 3 años?
4. ¿Qué tipo de café (arábica / robusta) predomina en esos mercados?

## 1.2 Evaluación de la situación

**Recursos disponibles**
| Recurso | Detalle |
|---|---|
| Consumo doméstico | 55 países productores, años cafeteros 1990/91 – 2019/20 |
| Producción total | Mismos 55 países y periodo (fuente pública, datos ICO) |
| Tipo de café | Arábica, robusta o mezcla, por país |

**Restricciones**
- **Sin precios.** El enunciado menciona "rangos de precios futuros", pero ningún dataset trae precios. Ese objetivo queda
  fuera del alcance y se reemplaza por proyección de volúmenes, que sí es posible con los datos. Integrar precios se propone para una iteración posterior.
- **Serie hasta 2019/20.** La proyección a 3 años cubre 2020/21 – 2022/23. No hay datos observados de esos años para validar,
  así que la evaluación se hace con backtesting sobre la historia.
- **Series anuales cortas** (30 puntos por país), lo que limita la complejidad de los modelos.
- **Plazo de entrega:** 2 días. Se prioriza lo simple, trazable y evaluable.

**Supuestos**
- La ICO es una fuente consistente entre archivos (se verifica en la fase 2).
- El balance producción − consumo es una buena aproximación de la necesidad de importación, aunque no incluye existencias ni re-exportaciones.

**Riesgos y contingencias**
| Riesgo | Contingencia |
|---|---|
| Datos faltantes o estimados por la fuente | Diagnóstico en 2.4 y reglas explícitas en la fase 3 |
| Producción muy volátil (clima, plagas, conflictos) | Proyectar con intervalos, no solo valores puntuales |
| Pocos datos para modelos complejos | Comparar siempre contra un modelo ingenuo |

**Terminología**
- **Año cafetero:** periodo de 12 meses de cosecha y comercialización. Cada país lo inicia en un mes distinto (abril, julio u octubre).
- **Saco:** unidad estándar de comercio, 60 kg de café verde.
- **Excedente exportable aparente:** producción − consumo doméstico.
- **Ratio consumo/producción:** fracción de la cosecha que absorbe el mercado interno. Mayor que 1 significa déficit.

## 1.3 Objetivos de analítica

1. Construir una tabla analítica país × año cafetero con consumo, producción, excedente y ratio en la misma unidad (sacos).
2. **Proyectar consumo y producción por país a 3 años cafeteros** con un modelo de series de tiempo y sus intervalos de confianza.
3. Derivar el excedente y el ratio proyectados.
4. Construir un **índice de prioridad** que combine:
   - **tamaño** del déficit proyectado,
   - **velocidad** con que se cierra el balance,
   - **certeza** de la proyección.

## 1.4 Criterios de éxito

**De negocio**
- Una lista priorizada y justificada de 5 a 10 mercados, entendible por el área comercial.
- Cada país priorizado tiene una explicación: por qué entra, cuánto volumen representa y con qué nivel de confianza.

**Técnicos**
- En backtesting con horizonte 3, el modelo elegido **supera al modelo ingenuo**: MASE < 1 en la mayoría de las series modelables.
- La cobertura de los intervalos de predicción es cercana a su nivel nominal (por ejemplo, ~80 % para un intervalo del 80 %).
- El ranking es **estable**: los primeros puestos no cambian drásticamente al cambiar de modelo candidato.
- El pipeline es reproducible de principio a fin desde los archivos de entrada.

## 1.5 Hipótesis de trabajo

| # | Hipótesis | Cómo se valida |
|---|---|---|
| H1 | El consumo doméstico de los productores crece más rápido que su producción | CAGR comparado (2.3) |
| H2 | Algunos productores ya son importadores netos | Excedente < 0 (fase 3) |
| H3 | La producción es mucho más volátil que el consumo, así que conviene proyectarlos por separado | Volatilidad comparada (2.3) |
| H4 | Un modelo de series de tiempo simple supera al ingenuo a 3 años | Backtesting (fases 4 y 5) |

## 1.6 Plan del proyecto

| Fase | Entregable |
|---|---|
| 2. Entendimiento de los datos | Diagnóstico de calidad y decisiones documentadas |
| 3. Preparación | `coffee_balance_long.parquet` y segmentación de series modelables |
| 4. Modelado | Torneo de modelos por serie (ingenuo, drift, ETS, ARIMA) con backtesting |
| 5. Evaluación | Métricas, índice de prioridad y ranking de mercados |
| 6. Despliegue | Presentación de resultados y propuesta de asistente GenAI |